<!-- KERNEL_BANNER -->
> **Use kernel: `mrigi_tor190_v8`**
>
> Set the notebook kernel to *Python (mrigi_tor190_v8)* before running.

# 30g: Fix the answer-extraction bug and re-judge

## The bug

23j/23k store the **entire prompt** in `model_answer`, not the model's answer.
`extract_completion()` splits on the tokenised tag
`<|start_header_id|>assistant<|end_header_id|>`, but the HuggingFace pipeline decodes
special tokens away before that string is ever seen — so the tag appears in **0 of 100**
records for every model. When the split target is absent the function returns its input
unchanged, so prompt + context + answer was stored and judged as "the response".

Measured contamination — the fraction of what the judge read that was actually the answer:

| Condition | Answer as % of judged text |
|---|---|
| no context | ~51% (rest is the instruction + question) |
| **RAG (MMR k=15)** | **~3.4%** (rest is 15 retrieved chunks) |

**Both metrics are affected**, completeness more than correctness:

| | r(contamination, correctness) | r(contamination, completeness) |
|---|---|---|
| no context | 0.14 – 0.27 | 0.35 – 0.45 |
| RAG | 0.20 – 0.36 | 0.40 – 0.52 |

All significant at p < 1e-5. This plausibly explains why *no context* appeared to beat
*RAG* on completeness: a 15,000-character wall of retrieved passages reads as unfocused
to a gold-blind judge, and the effect was largest for the models whose real answers were
shortest.

## What this notebook does

1. **Re-extract** every answer with a prompt-anchored extractor (validated at 96.4%
   recovery, 0 prompt leaks; the rest were genuinely empty generations).
2. **Write** a corrected results file — the GPU generations are reused, nothing is re-run.
3. **Re-judge** correctness and completeness on the clean answers.
4. **Compare** old vs new and report which conclusions change.

No model inference is re-run; only judging.

In [1]:
import os, json, glob, time, httpx
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.stats import wilcoxon

with open('/home/jupyter/Mrigi/env.sh') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            k, v = line[len('export '):].split('=', 1)
            os.environ[k] = v.strip('"').strip("'")

KEY = os.environ['OPENAI_API_TOKEN']
HEADERS = {'Authorization': f'Bearer {KEY}', 'Content-Type': 'application/json'}
API = 'https://api.openai.com/v1/chat/completions'

JUDGES  = ['gpt-4o']          # set to ['gpt-4o', 'gpt-4.1'] to re-judge with both
TASKS   = ['correctness', 'completeness']
WORKERS = 8
TS      = datetime.now().strftime('%Y%m%d_%H%M%S')
SRC     = 'results_23j_checkpoint.json'

# Only the 9 Table 1 variants are re-judged.
TABLE1 = ['Llama-3-8B-Instruct', 'base_llama_COT', 'DAPT_LR1e5', 'DAPT_LR1e5_COT',
          'synv2V2_step80', 'synv2V2_step80_COT', 'synv2_base_step80_COT',
          'fullpaper_120M_LR1e5', 'fullpaper_120M_COT']

probe = httpx.post(API, headers=HEADERS, timeout=60,
                   json={'model': 'gpt-4o-mini',
                         'messages': [{'role': 'user', 'content': 'hi'}], 'max_tokens': 1})
if probe.status_code != 200:
    raise RuntimeError(f'API unusable (HTTP {probe.status_code}): '
                       f'{probe.json().get("error", {}).get("message", "")[:160]}')
print(f'\u2713 billing live   judges={JUDGES}   tasks={TASKS}')

✓ billing live   judges=['gpt-4o']   tasks=['correctness', 'completeness']


## Step 1 — Re-extract the answers

In [2]:
def extract_answer(stored):
    """Strip the echoed prompt from a stored `model_answer`.

    Both generation prompts terminate with the literal 'Answer:'. Instruct models then
    emit a bare 'assistant' turn marker (the special tokens having been decoded away);
    base LMs emit nothing and simply continue. Splitting on the LAST 'Answer:' handles
    both, and is safe because the retrieved-context block never ends with that string.
    """
    if not isinstance(stored, str) or stored in ('ERROR', 'INVALID'):
        return None
    i = stored.rfind('Answer:')
    tail = (stored[i + len('Answer:'):] if i != -1 else stored).lstrip()
    if tail.startswith('assistant'):
        tail = tail[len('assistant'):].lstrip()
    for tok in ('<|eot_id|>', '<|end_of_text|>', '<|begin_of_text|>',
                '<|start_header_id|>', '<|end_header_id|>'):
        tail = tail.replace(tok, '')
    return tail.strip()


gen = json.load(open(SRC))
PROMPT_FRAGMENT = 'You are an expert on zeolite synthesis'

clean, report = {}, []
for m in TABLE1:
    clean[m] = {}
    for cond in ('no_context', 'mmr'):
        out = []
        for r in gen[m][cond]['detailed_results']:
            a = extract_answer(r.get('model_answer'))
            out.append({
                'query': r.get('query'),
                'gold': r.get('correct_answer_text'),
                'title': r.get('title'), 'doi': r.get('doi'),
                'answer': a if a else 'INVALID',
                'stored_len': len(r.get('model_answer') or ''),
                'answer_len': len(a) if a else 0,
            })
        clean[m][cond] = out
        ok = [o for o in out if o['answer'] != 'INVALID']
        leaks = sum(1 for o in ok if PROMPT_FRAGMENT in o['answer'])
        report.append({'model': m, 'condition': cond, 'n': len(out), 'recovered': len(ok),
                       'empty': len(out) - len(ok), 'prompt_leaks': leaks,
                       'mean_stored': np.mean([o['stored_len'] for o in out]),
                       'mean_answer': np.mean([o['answer_len'] for o in ok]) if ok else 0})

rep = pd.DataFrame(report)
rep['pct_signal'] = (rep.mean_answer / rep.mean_stored * 100).round(1)
print(rep[['model', 'condition', 'recovered', 'n', 'empty', 'prompt_leaks',
           'mean_stored', 'mean_answer', 'pct_signal']].to_string(index=False))
print(f'\ntotal recovered : {rep.recovered.sum()}/{rep.n.sum()}')
print(f'prompt leaks    : {rep.prompt_leaks.sum()}  (must be ~0)')

path = f'results_23j_clean_{TS}.json'
with open(path, 'w') as f:
    json.dump({'source': SRC, 'timestamp': TS,
               'note': 'answers re-extracted; prompt/context stripped', 'results': clean}, f, indent=2)
print(f'\n\u2713 saved {path}')
print(f'\nBEFORE: {clean["DAPT_LR1e5"]["mmr"][3]["stored_len"]:,} chars stored')
print(f'AFTER : {clean["DAPT_LR1e5"]["mmr"][3]["answer"][:300]}')

                model  condition  recovered   n  empty  prompt_leaks  mean_stored  mean_answer  pct_signal
  Llama-3-8B-Instruct no_context        100 100      0             0      1145.49   570.940000        49.8
  Llama-3-8B-Instruct        mmr        100 100      0             0     14778.62   430.840000         2.9
       base_llama_COT no_context        100 100      0             0      1346.61   767.040000        57.0
       base_llama_COT        mmr        100 100      0             0     15234.84   687.040000         4.5
           DAPT_LR1e5 no_context        100 100      0             0      1040.81   466.260000        44.8
           DAPT_LR1e5        mmr        100 100      0             0     14622.58   274.800000         1.9
       DAPT_LR1e5_COT no_context        100 100      0             0      1312.15   733.610000        55.9
       DAPT_LR1e5_COT        mmr        100 100      0             0     15174.75   627.130000         4.1
       synv2V2_step80 no_context     

## Step 2 — Re-judge on the clean answers

Prompts are verbatim from 30b, so old and new scores are directly comparable; the only
thing that changed is the `{response}` slot.

In [3]:
JUDGE_SYSTEM_PROMPT = """You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system 
focused on zeolite synthesis, catalysis, and environmental applications. 
You will evaluate the quality of retrieved contexts and open-ended generated responses 
against a reference (gold) answer.
Always respond in valid JSON format."""

CORRECTNESS = """Given the following query, a reference (gold) answer, and a model's open-ended response, rate how correct the response is.

Judge on scientific substance, not wording. The response may phrase things differently, be more or less detailed, or add related information \u2014 that is fine as long as the core claim matches the reference and any additional claims are accurate.

**Scoring Rubric (1-10):**
- 1-2: Completely incorrect \u2014 asserts something that contradicts the reference answer
- 3-4: Mostly incorrect \u2014 misses the main point, but shows partial understanding of the topic
- 5-6: Partially correct \u2014 captures part of the reference answer but is missing or muddling the key idea
- 7-8: Mostly correct \u2014 captures the key idea of the reference answer, with minor gaps or imprecise phrasing
- 9-10: Fully correct \u2014 captures the reference answer's key idea clearly and accurately (extra correct detail is fine)

**Query:**
{query}

**Reference (Gold) Answer:**
{gold_answer}

**Model's Open-Ended Response:**
{response}

Respond with ONLY a JSON object in this exact format:
{{"score": <integer 1-10>, "reasoning": "<one to two sentences explaining your score>"}}"""

COMPLETENESS = """Given the following query and a model's open-ended response, rate how completely the response addresses the question.

Judge how much of what the question is asking for is present \u2014 the main mechanism, the reason, or the required piece of reasoning. Do not penalize concise answers if they cover the essential point; do penalize hand-wavy or partial answers that skip the substantive part.

**Scoring Rubric (1-10):**
- 1-2: No meaningful response \u2014 empty, garbled, refuses, or completely off-topic
- 3-4: Minimal response \u2014 mentions the topic but does not really answer the question
- 5-6: Partial response \u2014 addresses part of the question but leaves the key aspect unexplained
- 7-8: Good response \u2014 addresses the question with a clear, relevant explanation of the main point
- 9-10: Excellent response \u2014 fully addresses the question with a thorough, well-reasoned explanation

**Query:**
{query}

**Model's Open-Ended Response:**
{response}

Respond with ONLY a JSON object in this exact format:
{{"score": <integer 1-10>, "reasoning": "<one to two sentences explaining your score>"}}"""


def call_judge(model, user, retries=4):
    body = {'model': model,
            'messages': [{'role': 'system', 'content': JUDGE_SYSTEM_PROMPT},
                         {'role': 'user', 'content': user}],
            'temperature': 0.1, 'max_tokens': 200,
            'response_format': {'type': 'json_object'}}
    for a in range(retries):
        try:
            r = httpx.post(API, headers=HEADERS, json=body, timeout=120)
        except Exception:
            time.sleep(3 * (a + 1)); continue
        if r.status_code == 200:
            try:
                d = json.loads(r.json()['choices'][0]['message']['content'].strip())
                s = int(d.get('score', -1))
                return (s if 1 <= s <= 10 else -1), d.get('reasoning', '')
            except Exception as e:
                return -1, f'parse: {e}'
        msg = r.json().get('error', {}).get('message', '')
        if r.status_code == 429 and 'credit' in msg.lower():
            return -1, 'billing'
        time.sleep(4 * (a + 1))
    return -1, 'retries exceeded'


plan = [(m, c, i) for m in TABLE1 for c in ('no_context', 'mmr')
        for i in range(len(clean[m][c]))]
print(f'{len(plan):,} records \u00d7 {len(TASKS)} tasks \u00d7 {len(JUDGES)} judge(s) '
      f'= {len(plan)*len(TASKS)*len(JUDGES):,} calls')


def judge_record(key):
    m, c, i = key
    rec = clean[m][c][i]
    out = {}
    if rec['answer'] == 'INVALID':
        for j in JUDGES:
            out[j] = {t: {'score': -1, 'reasoning': 'empty generation'} for t in TASKS}
        return key, out
    for j in JUDGES:
        out[j] = {}
        for t in TASKS:
            tmpl = CORRECTNESS if t == 'correctness' else COMPLETENESS
            user = (tmpl.format(query=rec['query'], gold_answer=rec['gold'], response=rec['answer'])
                    if t == 'correctness'
                    else tmpl.format(query=rec['query'], response=rec['answer']))
            s, why = call_judge(j, user)
            out[j][t] = {'score': s, 'reasoning': why}
    return key, out


new_scores = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(judge_record, k) for k in plan]
    for fut in tqdm(as_completed(futs), total=len(futs), desc='re-judging clean answers'):
        k, v = fut.result(); new_scores[k] = v

out_path = f'judge_results_23j_clean_{TS}.json'
with open(out_path, 'w') as f:
    json.dump({'metadata': {'source': path, 'judges': JUDGES, 'tasks': TASKS,
                            'schema': 'clean-extraction-v1', 'timestamp': TS,
                            'note': 'judged on prompt-stripped answers'},
               'results': {f'{m}|{c}|{i}': v for (m, c, i), v in new_scores.items()}}, f, indent=2)
print(f'\u2713 saved {out_path}')

1,800 records × 2 tasks × 1 judge(s) = 3,600 calls


re-judging clean answers: 100%|██████████| 1800/1800 [08:34<00:00,  3.50it/s]

✓ saved judge_results_23j_clean_20260803_194431.json


## Step 3 — Old vs new: what actually changed

In [4]:
OLD = json.load(open(sorted(glob.glob('judge_results_23j_dual_final_*.json'),
                            key=os.path.getmtime)[-1]))['results']
J = JUDGES[0]

rows = []
for m in TABLE1:
    for c in ('no_context', 'mmr'):
        old_recs = OLD[m].get(c) or []
        for t in TASKS:
            o = [r['judges'][J][f'{t}_score'] for r in old_recs
                 if r and r['judges'][J][f'{t}_score'] > 0]
            n = [new_scores[(m, c, i)][J][t]['score'] for i in range(len(clean[m][c]))
                 if new_scores[(m, c, i)][J][t]['score'] > 0]
            paired = [(new_scores[(m, c, i)][J][t]['score'],
                       old_recs[i]['judges'][J][f'{t}_score'])
                      for i in range(min(len(clean[m][c]), len(old_recs)))
                      if old_recs[i] and old_recs[i]['judges'][J][f'{t}_score'] > 0
                      and new_scores[(m, c, i)][J][t]['score'] > 0]
            d = np.array([a - b for a, b in paired], float)
            rows.append({'model': m, 'condition': c, 'task': t,
                         'old': np.mean(o) if o else np.nan,
                         'new': np.mean(n) if n else np.nan,
                         'delta': d.mean() if len(d) else np.nan,
                         'p': wilcoxon(d).pvalue if len(d) > 2 and np.any(d) else np.nan,
                         'n': len(d)})
cmp = pd.DataFrame(rows)
cmp.to_csv(f'extraction_fix_impact_{TS}.csv', index=False)

for t in TASKS:
    print(f'\n=== {t.upper()}  (judge {J})   old \u2192 new ===')
    print(f'{"model":<26}{"no context":>26}{"RAG":>26}')
    print(f'{"":<26}{"old":>8}{"new":>8}{"delta":>9}{"old":>9}{"new":>8}{"delta":>9}')
    print('-' * 78)
    for m in TABLE1:
        a = cmp[(cmp.model == m) & (cmp.condition == 'no_context') & (cmp.task == t)].iloc[0]
        b = cmp[(cmp.model == m) & (cmp.condition == 'mmr') & (cmp.task == t)].iloc[0]
        print(f'{m:<26}{a.old:>8.2f}{a.new:>8.2f}{a.delta:>+9.2f}'
              f'{b.old:>9.2f}{b.new:>8.2f}{b.delta:>+9.2f}')

print('\n=== Does RAG still lose to no-context on completeness? ===')
for m in TABLE1:
    o_nc = cmp[(cmp.model == m) & (cmp.condition == 'no_context') & (cmp.task == 'completeness')].iloc[0]
    o_r  = cmp[(cmp.model == m) & (cmp.condition == 'mmr') & (cmp.task == 'completeness')].iloc[0]
    print(f'  {m:<26} BEFORE {o_nc.old - o_r.old:+.2f}   AFTER {o_nc.new - o_r.new:+.2f}'
          f'   {"(sign flipped)" if np.sign(o_nc.old-o_r.old) != np.sign(o_nc.new-o_r.new) else ""}')

print(f'\nMean absolute change: correctness '
      f'{cmp[cmp.task=="correctness"].delta.abs().mean():.2f}, completeness '
      f'{cmp[cmp.task=="completeness"].delta.abs().mean():.2f}')
print(f'saved extraction_fix_impact_{TS}.csv')


=== CORRECTNESS  (judge gpt-4o)   old → new ===
model                                     no context                       RAG
                               old     new    delta      old     new    delta
------------------------------------------------------------------------------
Llama-3-8B-Instruct           7.15    7.41    +0.26     7.56    7.41    -0.16
base_llama_COT                7.58    7.94    +0.35     7.73    8.08    +0.32
DAPT_LR1e5                    6.96    7.26    +0.30     6.65    6.52    -0.14
DAPT_LR1e5_COT                7.30    7.68    +0.38     7.76    8.04    +0.28
synv2V2_step80                7.05    7.33    +0.28     7.01    6.89    -0.13
synv2V2_step80_COT            7.44    7.76    +0.32     7.83    8.21    +0.38
synv2_base_step80_COT         4.92    4.97    +0.09     7.51    7.53    +0.15
fullpaper_120M_LR1e5          6.74    7.08    +0.34     6.74    6.48    -0.27
fullpaper_120M_COT            7.31    7.83    +0.42     7.79    8.13    +0.41

=== COMPLETE

In [5]:
# Re-check the headline findings on the corrected scores.
PAIRS = [('Llama-3-8B-Instruct', 'base_llama_COT', 'Llama-3-8B-Instruct (no DAPT)'),
         ('DAPT_LR1e5', 'DAPT_LR1e5_COT', 'Broad abstracts DAPT'),
         ('synv2V2_step80', 'synv2V2_step80_COT', 'Synthesis abstracts DAPT'),
         ('fullpaper_120M_LR1e5', 'fullpaper_120M_COT', 'Synthesis full-paper DAPT')]


def newmean(m, c, t):
    v = [new_scores[(m, c, i)][J][t]['score'] for i in range(len(clean[m][c]))
         if new_scores[(m, c, i)][J][t]['score'] > 0]
    return np.mean(v) if v else np.nan


def stars(p):
    return '' if not np.isfinite(p) else '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < .05 else 'n.s.'


print('FINDING 1 - CoT gain, on corrected scores')
for t in TASKS:
    print(f'  [{t}]')
    for b, ct, lab in PAIRS:
        line = f'    {lab:<32}'
        for c in ('no_context', 'mmr'):
            d = np.array([new_scores[(ct, c, i)][J][t]['score'] - new_scores[(b, c, i)][J][t]['score']
                          for i in range(len(clean[b][c]))
                          if new_scores[(ct, c, i)][J][t]['score'] > 0
                          and new_scores[(b, c, i)][J][t]['score'] > 0], float)
            p = wilcoxon(d).pvalue if len(d) > 2 and np.any(d) else np.nan
            line += f'  {c}: {d.mean():+.2f} {stars(p):<4}'
        print(line)

print('\nFINDING 2 - does retrieval help each model? (corrected correctness)')
for m in TABLE1:
    a, b = newmean(m, 'no_context', 'correctness'), newmean(m, 'mmr', 'correctness')
    tag = 'CoT ' if 'COT' in m else 'base'
    print(f'  {m:<26}{tag}  {a:.2f} \u2192 {b:.2f}   {b-a:+.2f}')

print('\nFINDING 3 - best configuration under RAG (corrected)')
for m, v in sorted(((m, newmean(m, 'mmr', 'correctness')) for m in TABLE1),
                   key=lambda x: -x[1])[:4]:
    print(f'  {v:.2f}  {m}')

print('\n' + '=' * 70)
print('Corrected scores live in judge_results_23j_clean_*.json.')
print('Point 30e at that file (and README numbers) once these look right.')
print('=' * 70)

FINDING 1 - CoT gain, on corrected scores
  [correctness]
    Llama-3-8B-Instruct (no DAPT)     no_context: +0.53 **    mmr: +0.67 *** 
    Broad abstracts DAPT              no_context: +0.42 *     mmr: +1.52 *** 
    Synthesis abstracts DAPT          no_context: +0.43 *     mmr: +1.32 *** 
    Synthesis full-paper DAPT         no_context: +0.75 ***   mmr: +1.65 *** 
  [completeness]
    Llama-3-8B-Instruct (no DAPT)     no_context: +0.36 ***   mmr: +0.74 *** 
    Broad abstracts DAPT              no_context: +0.34 ***   mmr: +1.92 *** 
    Synthesis abstracts DAPT          no_context: +0.33 ***   mmr: +1.61 *** 
    Synthesis full-paper DAPT         no_context: +0.59 ***   mmr: +1.81 *** 

FINDING 2 - does retrieval help each model? (corrected correctness)
  Llama-3-8B-Instruct       base  7.41 → 7.41   +0.00
  base_llama_COT            CoT   7.94 → 8.08   +0.14
  DAPT_LR1e5                base  7.26 → 6.52   -0.74
  DAPT_LR1e5_COT            CoT   7.68 → 8.04   +0.36
  synv2V2_step80

In [6]:
# ---------------------------------------------------------------------------
# GPT-5.2, re-judged under the same clean protocol.
#
# Its answers came back through the API and were never affected by the extraction
# bug, so nothing needs stripping. But two things must match the open-weight models
# for the comparison to be fair: (a) the same judge and prompts, and (b) the same
# question set - so if 23L changed Q67, that one item is regenerated here.
#
# Reported as a numeric reference only; deliberately NOT plotted (see README: the
# GPT-5.2 comparison belongs in the SI, not the main paper).
# ---------------------------------------------------------------------------
JUDGE_GPT52 = True

if JUDGE_GPT52:
    gfiles = sorted(glob.glob('results_gpt52_nocontext_*.json'), key=os.path.getmtime)
    if not gfiles:
        print('No GPT-5.2 generation found - run 30f first. Skipping.')
    else:
        g52 = json.load(open(gfiles[-1]))['detailed_results']
        print(f'loaded {gfiles[-1]}  ({len(g52)} answers)')

        # (b) question-set parity against the canonical set used by the open-weight models
        canonical = [clean[TABLE1[0]]['no_context'][i]['query'] for i in range(len(clean[TABLE1[0]]['no_context']))]
        mismatch = [i for i, r in enumerate(g52)
                    if (r.get('query') or '').strip() != (canonical[i] or '').strip()]
        print(f'question mismatches vs canonical set: {mismatch if mismatch else "none"}')

        NO_CTX_PROMPT = ("You are an expert on zeolite synthesis, chemistry, and catalysis. "
                         "Answer the following question using your own knowledge. Give a clear, "
                         "focused answer in 3\u20135 sentences (or fewer if the question is simple). "
                         "Do not invent citations. Do not restate the question.\n\n"
                         "Question: {question}\n\nAnswer:")

        for i in mismatch:
            q = canonical[i]
            gold = clean[TABLE1[0]]['no_context'][i]['gold']
            body = {'model': 'gpt-5.2',
                    'messages': [{'role': 'user', 'content': NO_CTX_PROMPT.format(question=q)}],
                    'max_completion_tokens': 400}
            r = httpx.post(API, headers=HEADERS, json=body, timeout=180)
            if r.status_code != 200:   # older param spelling
                body['max_tokens'] = body.pop('max_completion_tokens')
                r = httpx.post(API, headers=HEADERS, json=body, timeout=180)
            txt = (r.json()['choices'][0]['message']['content'] or '').strip() if r.status_code == 200 else 'ERROR'
            g52[i] = {'q_idx': i, 'query': q, 'gold': gold, 'model_answer': txt}
            print(f'   regenerated q{i}: {txt[:90]}...')

        if mismatch:
            newp = f'results_gpt52_nocontext_{TS}.json'
            json.dump({'model': 'GPT-5.2', 'model_id': 'gpt-5.2', 'method': 'no_context',
                       'timestamp': TS, 'n_valid': sum(1 for r in g52 if r['model_answer'] not in ('ERROR','INVALID')),
                       'total': len(g52), 'detailed_results': g52}, open(newp, 'w'), indent=2)
            print(f'   saved {newp}')

        def judge_g52(i):
            rec = g52[i]
            if rec.get('model_answer') in ('ERROR', 'INVALID'):
                return i, {j: {t: {'score': -1, 'reasoning': 'invalid'} for t in TASKS} for j in JUDGES}
            out = {}
            for j in JUDGES:
                out[j] = {}
                for t in TASKS:
                    user = (CORRECTNESS.format(query=rec['query'], gold_answer=rec['gold'],
                                               response=rec['model_answer']) if t == 'correctness'
                            else COMPLETENESS.format(query=rec['query'], response=rec['model_answer']))
                    s, why = call_judge(j, user)
                    out[j][t] = {'score': s, 'reasoning': why}
            return i, out

        g52_scores = {}
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futs = [ex.submit(judge_g52, i) for i in range(len(g52))]
            for fut in tqdm(as_completed(futs), total=len(futs), desc='judging GPT-5.2 (clean)'):
                i, v = fut.result(); g52_scores[i] = v

        json.dump({'model': 'GPT-5.2', 'judges': JUDGES, 'tasks': TASKS,
                   'schema': 'clean-extraction-v1', 'timestamp': TS,
                   'results': {str(i): v for i, v in g52_scores.items()}},
                  open(f'judge_results_gpt52_clean_{TS}.json', 'w'), indent=2)

        print('\n' + '=' * 74)
        print('GPT-5.2 (no context) on the corrected, unified question set')
        print('=' * 74)
        for j in JUDGES:
            for t in TASKS:
                v = [g52_scores[i][j][t]['score'] for i in g52_scores if g52_scores[i][j][t]['score'] > 0]
                print(f'  [{j}] {t:<13} {np.mean(v):.2f}   (n={len(v)})')

        print('\n  vs open-weight models, no-context, same judge/questions:')
        for t in TASKS:
            gv = {i: g52_scores[i][J][t]['score'] for i in g52_scores if g52_scores[i][J][t]['score'] > 0}
            print(f'    [{t}]')
            for m in TABLE1:
                lv = {i: new_scores[(m, 'no_context', i)][J][t]['score']
                      for i in range(len(clean[m]['no_context']))
                      if new_scores[(m, 'no_context', i)][J][t]['score'] > 0}
                k = sorted(set(gv) & set(lv))
                d = np.array([gv[i] - lv[i] for i in k], float)
                p = wilcoxon(d).pvalue if len(d) > 2 and np.any(d) else np.nan
                print(f'      {m:<26}{np.mean([lv[i] for i in k]):>6.2f}   '
                      f'GPT-5.2 {d.mean():+.2f} {stars(p):<4} (W/L/T {int((d>0).sum())}/{int((d<0).sum())}/{int((d==0).sum())})')
        print('\n  (numeric reference only - not plotted; SI material)')


loaded results_gpt52_nocontext_20260803_004922.json  (100 answers)
question mismatches vs canonical set: [67]
   regenerated q67: In very Si‑rich SSZ‑13, Al atoms are often isolated or separated such that no local region...
   saved results_gpt52_nocontext_20260803_194431.json


judging GPT-5.2 (clean): 100%|██████████| 100/100 [00:29<00:00,  3.40it/s]


GPT-5.2 (no context) on the corrected, unified question set
  [gpt-4o] correctness   9.14   (n=100)
  [gpt-4o] completeness  9.03   (n=100)

  vs open-weight models, no-context, same judge/questions:
    [correctness]
      Llama-3-8B-Instruct         7.41   GPT-5.2 +1.73 ***  (W/L/T 76/0/24)
      base_llama_COT              7.94   GPT-5.2 +1.20 ***  (W/L/T 63/1/36)
      DAPT_LR1e5                  7.26   GPT-5.2 +1.88 ***  (W/L/T 76/1/23)
      DAPT_LR1e5_COT              7.68   GPT-5.2 +1.46 ***  (W/L/T 67/1/32)
      synv2V2_step80              7.33   GPT-5.2 +1.81 ***  (W/L/T 74/0/26)
      synv2V2_step80_COT          7.76   GPT-5.2 +1.38 ***  (W/L/T 69/0/31)
      synv2_base_step80_COT       4.97   GPT-5.2 +4.17 ***  (W/L/T 88/0/11)
      fullpaper_120M_LR1e5        7.08   GPT-5.2 +2.06 ***  (W/L/T 77/1/22)
      fullpaper_120M_COT          7.83   GPT-5.2 +1.31 ***  (W/L/T 67/0/33)
    [completeness]
      Llama-3-8B-Instruct         8.63   GPT-5.2 +0.40 ***  (W/L/T 37/0/63)
  

In [7]:
# ---- add gpt-4.1 to the existing clean judgments (reuses the live kernel) ----
EXTRA_JUDGE = 'gpt-4.1'

def judge_extra(key):
    m, c, i = key
    rec = clean[m][c][i]
    if rec['answer'] == 'INVALID':
        return key, {t: {'score': -1, 'reasoning': 'empty generation'} for t in TASKS}
    out = {}
    for t in TASKS:
        user = (CORRECTNESS.format(query=rec['query'], gold_answer=rec['gold'], response=rec['answer'])
                if t == 'correctness'
                else COMPLETENESS.format(query=rec['query'], response=rec['answer']))
        s, why = call_judge(EXTRA_JUDGE, user)
        out[t] = {'score': s, 'reasoning': why}
    return key, out

with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(judge_extra, k) for k in plan]
    for fut in tqdm(as_completed(futs), total=len(futs), desc=f'judging with {EXTRA_JUDGE}'):
        k, v = fut.result()
        new_scores[k][EXTRA_JUDGE] = v          # merge alongside gpt-4o

JUDGES_ALL = JUDGES + [EXTRA_JUDGE]
TS2 = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path2 = f'judge_results_23j_clean_{TS2}.json'
with open(out_path2, 'w') as f:
    json.dump({'metadata': {'source': path, 'judges': JUDGES_ALL, 'tasks': TASKS,
                            'schema': 'clean-extraction-v1', 'timestamp': TS2,
                            'note': 'judged on prompt-stripped answers; both judges'},
               'results': {f'{m}|{c}|{i}': v for (m, c, i), v in new_scores.items()}}, f, indent=2)
print(f'saved {out_path2}  (judges: {JUDGES_ALL})')


judging with gpt-4.1: 100%|██████████| 1800/1800 [13:14<00:00,  2.27it/s]

saved judge_results_23j_clean_20260803_201225.json  (judges: ['gpt-4o', 'gpt-4.1'])


In [8]:
# ---- add gpt-4.1 to the GPT-5.2 judgments ----
def judge_g52_extra(i):
    rec = g52[i]
    if rec.get('model_answer') in ('ERROR', 'INVALID'):
        return i, {t: {'score': -1, 'reasoning': 'invalid'} for t in TASKS}
    out = {}
    for t in TASKS:
        user = (CORRECTNESS.format(query=rec['query'], gold_answer=rec['gold'],
                                   response=rec['model_answer']) if t == 'correctness'
                else COMPLETENESS.format(query=rec['query'], response=rec['model_answer']))
        s, why = call_judge(EXTRA_JUDGE, user)
        out[t] = {'score': s, 'reasoning': why}
    return i, out

with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(judge_g52_extra, i) for i in range(len(g52))]
    for fut in tqdm(as_completed(futs), total=len(futs), desc=f'GPT-5.2 with {EXTRA_JUDGE}'):
        i, v = fut.result()
        g52_scores[i][EXTRA_JUDGE] = v

with open(f'judge_results_gpt52_clean_{TS2}.json', 'w') as f:
    json.dump({'model': 'GPT-5.2', 'judges': JUDGES_ALL, 'tasks': TASKS,
               'schema': 'clean-extraction-v1', 'timestamp': TS2,
               'results': {str(i): v for i, v in g52_scores.items()}}, f, indent=2)

for j in JUDGES_ALL:
    for t in TASKS:
        v = [g52_scores[i][j][t]['score'] for i in g52_scores if g52_scores[i][j][t]['score'] > 0]
        print(f'  GPT-5.2 [{j}] {t:<13} {np.mean(v):.2f}  (n={len(v)})')


GPT-5.2 with gpt-4.1: 100%|██████████| 100/100 [00:45<00:00,  2.21it/s]

  GPT-5.2 [gpt-4o] correctness   9.14  (n=100)
  GPT-5.2 [gpt-4o] completeness  9.03  (n=100)
  GPT-5.2 [gpt-4.1] correctness   9.91  (n=100)
  GPT-5.2 [gpt-4.1] completeness  10.00  (n=100)


In [9]:
from scipy.stats import pearsonr
print(f'{"task":<14}{"r":>8}{"within ±1":>12}{"mean |diff|":>13}{"n":>7}')
for t in TASKS:
    a, b = [], []
    for k in plan:
        x = new_scores[k]['gpt-4o'][t]['score']; y = new_scores[k]['gpt-4.1'][t]['score']
        if x > 0 and y > 0: a.append(x); b.append(y)
    a, b = np.array(a), np.array(b)
    print(f'{t:<14}{pearsonr(a,b)[0]:>8.3f}{np.mean(np.abs(a-b)<=1)*100:>11.0f}%'
          f'{np.mean(np.abs(a-b)):>13.2f}{len(a):>7}')

print('\nBest configuration under each judge (RAG correctness, corrected):')
for j in JUDGES_ALL:
    r = sorted(((np.mean([new_scores[(m,'mmr',i)][j]['correctness']['score']
                          for i in range(len(clean[m]['mmr']))
                          if new_scores[(m,'mmr',i)][j]['correctness']['score'] > 0]), m)
                for m in TABLE1), reverse=True)
    print(f'  {j}: ' + '  >  '.join(f'{m} ({v:.2f})' for v, m in r[:3]))


task                 r   within ±1  mean |diff|      n
correctness      0.913         95%         0.59   1790
completeness     0.874         92%         0.55   1790

Best configuration under each judge (RAG correctness, corrected):
  gpt-4o: synv2V2_step80_COT (8.21)  >  fullpaper_120M_COT (8.13)  >  base_llama_COT (8.08)
  gpt-4.1: base_llama_COT (8.76)  >  synv2V2_step80_COT (8.75)  >  fullpaper_120M_COT (8.57)
